1 Load the Dataset

In [ ]:
!pip install dataset

In [ ]:
!pip install -U dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

2 Tokenize the dataset 

2.1 Tokenize the dataset into tokenIDs
2.2 Create a file called "train.bin" and "validation.bin" where we will store the tokenIDs for the entire dataset.
2.3 We make sure the tokenIDs are stored on a disk, rather than on RAM for efficient computation.

In [ ]:
!pip install tiktoken
import tiktoken
import os 
import numpy as np

from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

#  Some functions from https://github.com/karpathy/nanoGPT/blob/master/data/openwebtext/prepare.py

def process(example):
    # encode_ordinary ignores any special token
    ids = enc.encode_ordinary(example["text"])
    out = {{"ids":ids, "len": len(ids)}}

    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=["text"],
        desc="tokeninzing the splits",
        num_proc=8,
    )

    # concatenate all the ids in each dataset into one large file we use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = f"{split}.bin"
        dtype = np.uint16 # can do since enc.max_token_value == 50256 is < 2**16
        arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f"writing {filename}"):
            # batch together samples for faster write
            batch = dset.shard(num_shard=total_batches, index=batch_idx, contiguous=True)
            arr_batch = np.concatenate(batch["ids"])
            # write into map 
            arr[idx: idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()

3. Creating Input-Output batches for the dataset

In [ ]:
# Some functions from https://github.com/karpathy/nanoGPT/blob/master/train.py with slight modifications

#block size = context window

def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per 
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122

    if split == "train":
        data = np.memmap("train.bin", dtype=np.uint16, mode="r")
    else:
        data = np.memmap("validation.bin", dtype=np.uint16, mode="r")

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    if device_type == "cuda":
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(
            device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

Define the SLM model architecture 